In [1]:
import pandas as pd
import numpy as np

In [2]:
train = pd.read_pickle("../data/train_cleaned.pkl")
train.shape

(84916, 9)

In [3]:
train.head()

,productid,imageid,designation,description,text_clean,has_description,word_count,char_count,prdtypecode
0,3804725264,1263597046,Olivia: Personalisiertes Notizbuch / 150 Seite...,NaN,olivia personalisiertes notizbuch seiten punkt...,0,10,74,10
1,436067568,1008141237,Journal Des Arts (Le) N° 133 Du 28/09/2001 - L...,NaN,journal des arts le n du l art et son marche s...,0,34,177,2280
2,201115110,938777978,Grand Stylet Ergonomique Bleu Gamepad Nintendo...,PILOT STYLE Touch Pen de marque Speedlink est ...,grand stylet ergonomique bleu gamepad nintendo...,1,118,731,50
3,50418756,457047496,Peluche Donald - Europe - Disneyland 2000 (Mar...,NaN,peluche donald europe disneyland marionnette a...,0,7,52,1280
4,278535884,1077757786,La Guerre Des Tuques,Luc a des id&eacute;es de grandeur. Il veut or...,la guerre des tuques luc a des id eacute es de...,1,46,227,2705


In [ ]:
total_len = []
for idx, row in train.iterrows():
    total_len.append(len(row["text_clean"]))
mean_len = np.mean(total_len)
print(mean_len)

#TODO: SUMMARIZE THE TEXT 
# text = row["text_clean"][:77]  # Truncate to 77 chars (CLIP limit)
# tok = tokenizer([text])
# txt_feat = clip_model.encode_text(tok)  # → 512-d vector
# txt_feat = txt_feat / txt_feat.norm()  # L2 normalize

523.765014838193


## Compare TF-IDF (baseline) vs TF-IDF with Translation

Same split (`random_state=42`, `test_size=0.20`, stratified), same LightGBM config, same metric (weighted F1).

In [ ]:
import pickle
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, f1_score, confusion_matrix

# ── Load data & reproduce the same split ─────────────────────────────────
train = pd.read_pickle("../data/train_cleaned.pkl")

le = LabelEncoder()
train["label"] = le.fit_transform(train["prdtypecode"])

X_train, X_val, y_train, y_val = train_test_split(
    train, train["label"],
    test_size=0.20, random_state=42, stratify=train["label"],
)
print(f"Val set: {len(X_val):,} rows, {le.classes_.shape[0]} classes")

### A) Baseline TF-IDF predictions

In [ ]:
# Load baseline artifacts
with open("../models/tfidf_vectorizer.pkl", "rb") as f:
    tfidf_base = pickle.load(f)
with open("../models/lgb_text_model.pkl", "rb") as f:
    model_base = pickle.load(f)

# Transform & predict
X_va_base = tfidf_base.transform(X_val["text_clean"])
y_pred_base = model_base.predict(X_va_base)
f1_base = f1_score(y_val, y_pred_base, average="weighted")

print(f"Baseline TF-IDF — weighted F1: {f1_base:.4f}")
print(classification_report(y_val, y_pred_base, target_names=[str(c) for c in le.classes_]))

### B) Translated TF-IDF predictions

> **Note:** Run `python models/tfidf_translated.py` first to generate the artifacts.

In [ ]:
# Load translated artifacts
with open("../models/tfidf_translated_vectorizer.pkl", "rb") as f:
    tfidf_trans = pickle.load(f)
with open("../models/lgb_translated_model.pkl", "rb") as f:
    model_trans = pickle.load(f)

# Need the translated text for the same val split
train_trans = pd.read_pickle("../data/train_translated.pkl")
train_trans["label"] = le.transform(train_trans["prdtypecode"])
_, X_val_trans, _, _ = train_test_split(
    train_trans, train_trans["label"],
    test_size=0.20, random_state=42, stratify=train_trans["label"],
)

X_va_trans = tfidf_trans.transform(X_val_trans["text_fr_clean"])
y_pred_trans = model_trans.predict(X_va_trans)
f1_trans = f1_score(y_val, y_pred_trans, average="weighted")

print(f"Translated TF-IDF — weighted F1: {f1_trans:.4f}")
print(classification_report(y_val, y_pred_trans, target_names=[str(c) for c in le.classes_]))

### C) Side-by-side comparison

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Summary table ────────────────────────────────────────────────────────
delta = f1_trans - f1_base
summary = pd.DataFrame({
    "Model": ["Baseline TF-IDF", "Translated TF-IDF"],
    "Weighted F1": [f1_base, f1_trans],
})
print(summary.to_string(index=False))
print(f"\nDelta (translated - baseline): {delta:+.4f} ({'improvement' if delta > 0 else 'regression'})")

# ── Per-class F1 comparison ──────────────────────────────────────────────
report_base = classification_report(y_val, y_pred_base, target_names=[str(c) for c in le.classes_], output_dict=True)
report_trans = classification_report(y_val, y_pred_trans, target_names=[str(c) for c in le.classes_], output_dict=True)

classes = [str(c) for c in le.classes_]
f1_per_class = pd.DataFrame({
    "class": classes,
    "f1_baseline": [report_base[c]["f1-score"] for c in classes],
    "f1_translated": [report_trans[c]["f1-score"] for c in classes],
})
f1_per_class["delta"] = f1_per_class["f1_translated"] - f1_per_class["f1_baseline"]
f1_per_class = f1_per_class.sort_values("delta", ascending=False)

# Bar chart: per-class F1 delta
fig_delta = go.Figure(go.Bar(
    x=f1_per_class["class"],
    y=f1_per_class["delta"],
    marker_color=f1_per_class["delta"].apply(lambda d: "green" if d > 0 else "red"),
    text=f1_per_class["delta"].apply(lambda d: f"{d:+.3f}"),
    textposition="outside",
))
fig_delta.update_layout(
    title="Per-Class F1 Change (Translated − Baseline)",
    xaxis_title="Product Category",
    yaxis_title="F1 Delta",
    height=500,
)
fig_delta.show()

In [ ]:
# ── Side-by-side grouped bar: F1 per class ───────────────────────────────
fig_compare = go.Figure()
fig_compare.add_trace(go.Bar(
    name="Baseline", x=f1_per_class["class"], y=f1_per_class["f1_baseline"],
    marker_color="steelblue",
))
fig_compare.add_trace(go.Bar(
    name="Translated", x=f1_per_class["class"], y=f1_per_class["f1_translated"],
    marker_color="coral",
))
fig_compare.update_layout(
    barmode="group",
    title="Per-Class F1: Baseline vs Translated TF-IDF",
    xaxis_title="Product Category",
    yaxis_title="F1-score",
    height=500,
    yaxis_range=[0, 1],
)
fig_compare.show()

# ── Show top improvers and regressors ────────────────────────────────────
print("Top 5 improved categories:")
print(f1_per_class.head(5)[["class", "f1_baseline", "f1_translated", "delta"]].to_string(index=False))
print("\nTop 5 regressed categories:")
print(f1_per_class.tail(5)[["class", "f1_baseline", "f1_translated", "delta"]].to_string(index=False))